In [ ]:
!pip install transformers
!pip install sentencepiece

## Local Inference on GPU 
Model page: https://huggingface.co/vesteinn/ScandiBERT

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/vesteinn/ScandiBERT)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("kaggle-scandibert")

import torch
from huggingface_hub import hf_hub_download
from transformers import XLMRobertaTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset, concatenate_datasets, get_dataset_config_names

model_name = "vesteinn/ScandiBERT" # Considering no-faroese variant for slight tokenization difference
labels = ["Norwegian", "Non-Scandinavian"]
id2label = {i: l for i, l in enumerate(labels)}
label2id = {l: i for i, l in enumerate(labels)}

# Hard-coded because tokenizer.json is broken
vocab_path = hf_hub_download(repo_id=model_name, filename="sentencepiece.bpe.model")
tokenizer = XLMRobertaTokenizer(vocab_file=vocab_path)


model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels = len(labels),
    id2label = id2label,  
    label2id = label2id,
    ignore_mismatched_sizes = True
)

dataset_norwegian = load_dataset("NbAiLab/nbnn_language_detection", split="train_nordic")
sample_norwegian_nb = dataset_norwegian.filter(lambda x: x["language"] == "nob").shuffle(seed=42).select(range(75))
sample_norwegian_nn = dataset_norwegian.filter(lambda x: x["language"] == "nno").shuffle(seed=42).select(range(25))
sample_norwegian_all = concatenate_datasets([sample_norwegian_nb, sample_norwegian_nn]).shuffle(seed=42)

nonscandi_configs = {  # For negative class of Non-Scandinavian languages that might pass regex
    "Spanish": ("en-es", "es"),
    "German": ("de-en", "de"),
    "Dutch": ("en-nl", "nl"),
    "Afrikaans": ("af-en", "af")
}

sample_nonscandi_all = []

for lang, (config, code) in nonscandi_configs.items(): # Collecting negative class sample, 100 sentences per language
    print(f"Downloading {lang} ({config}) for negative class...")
    dataset_nonscandi = load_dataset("Helsinki-NLP/opus-100", config, split="train")
    dataset_nonscandi = dataset_nonscandi.shuffle(seed=42)
    sample_nonscandi = dataset_nonscandi.select(range(100))
    sentences = [item[code] for item in sample_nonscandi["translation"]]
    sample_nonscandi_all.extend(sentences)
print(f"Non-Scandinavian corpora collected: {len(sample_nonscandi_all)} sentences.")

text = "Dette er en norsk setning."
inputs = tokenizer(text, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)
print("\n")
probs = torch.softmax(outputs.logits, dim=-1)
for label, prob in zip(labels, probs[0]):
    print(f"{label}: {prob.item():.4f}")

print("\n")
#for i in sample_norwegian_all["text"]:
    #print(i)
#for sent in sample_nonscandi_all[:5]:
    #print("-", sent)